# Minimum-Variance Hedge Ratio

How many futures to short against a position? Minimize the variance of the hedged position
with one derivative, and the answer turns out to be the beta from the previous notebook.

## 0. Math link

Two results from the sibling repos:

- Variance of a difference: [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb),
  applied in [01 Variance of a sum and a difference](./01_variance_of_sum_and_difference.ipynb).
- Minimizing a function with the first and second derivative: [Local extrema](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/10_local_extrema.ipynb).

## 1. Introduction

In 01 the hedge was always 1:1, short one unit of the hedge for each unit of exposure. That
was arbitrary, and it showed: with a weakly correlated hedge instrument the 1:1 hedge made
risk **worse**. In 02 we saw that beta, not correlation, measures how much one asset moves
with another.

This notebook makes the hedge size a choice variable $h$ and picks the one that minimizes risk.
It answers three practical questions:

- **How many contracts?** The hedge ratio $h^*$ and the number of futures $N^*$.
- **How good is the hedge?** The share of variance it removes, and the risk left over.
- **How reliable is it?** $h^*$ is estimated from data, and the estimate is noisy.

## 2. Theory

### The hedged position

Let $S$ be the return of the position to hedge (the spot) and $F$ the return of the hedge
instrument (a future). Short $h$ units of $F$ per unit of $S$:

$$
H(h) = S - h\,F
$$

### Variance as a function of $h$

From 01, with $a = 1$, $b = -h$:

$$
V(h) = \operatorname{Var}(S - hF) = \sigma_S^2 + h^2\sigma_F^2 - 2h\,\rho\,\sigma_S\sigma_F
$$

A parabola in $h$, opening upward.

### Minimize

First derivative equal to zero:

$$
V'(h) = 2h\,\sigma_F^2 - 2\rho\,\sigma_S\sigma_F = 0
\quad\Longrightarrow\quad
h^* = \rho\,\frac{\sigma_S}{\sigma_F} = \frac{\operatorname{Cov}(S,F)}{\operatorname{Var}(F)}
$$

Second derivative $V''(h) = 2\sigma_F^2 > 0$, so $h^*$ is a minimum, and the only one.

$h^*$ is exactly the **beta** of $S$ on $F$ from 02: the slope of the regression of spot
returns on futures returns.

### The minimum variance

Substitute $h^*$ back:

$$
V(h^*) = \sigma_S^2 + \rho^2\sigma_S^2 - 2\rho^2\sigma_S^2 = \sigma_S^2\,(1-\rho^2)
$$

- **Hedge effectiveness** (share of variance removed): $1 - V(h^*)/\sigma_S^2 = \rho^2$, the $R^2$ of the regression.
- **Residual volatility**: $\sigma_S\sqrt{1-\rho^2}$.

Correlation decides how **good** the hedge can be, the volatility ratio decides how **big** it must be.

### The residual is uncorrelated with the hedge

$$
\operatorname{Cov}(S - h^*F,\ F) = \operatorname{Cov}(S,F) - h^*\operatorname{Var}(F) = 0
$$

At the optimum, nothing left in the hedged position can be explained (linearly) by $F$. Any
other $h$ leaves some $F$ exposure in, which adds variance.

### From ratio to contracts

$h^*$ is per unit of value. With exposure value $Q_S$ and one futures contract worth $Q_F$
(index level times multiplier):

$$
N^* = h^*\,\frac{Q_S}{Q_F}
$$

## 3. Worked Examples

### Example 1: equal volatilities

$\sigma_S = \sigma_F = \sigma$, $\rho = 0.9$:

- $h^* = 0.9$, $V(h^*) = \sigma^2(1 - 0.81) = 0.19\,\sigma^2$.
- 1:1 hedge: $V(1) = 2\sigma^2(1-0.9) = 0.20\,\sigma^2$.

With equal volatilities the 1:1 hedge is nearly optimal when $\rho$ is high. The gap opens when
$\rho$ is low or the volatilities differ.

### Example 2: hedging a Swiss equity portfolio with SMI futures

Portfolio value CHF 10,000,000, annual volatility $\sigma_S = 18\%$. SMI futures: $\sigma_F = 15\%$,
$\rho = 0.95$. Assume the SMI stands at 12,000 points; the contract multiplier is CHF 10 per
index point, so one contract is worth CHF 120,000.

$$
h^* = 0.95 \cdot \frac{0.18}{0.15} = 1.14,
\qquad
N^* = 1.14 \cdot \frac{10{,}000{,}000}{120{,}000} = 95 \text{ contracts}
$$

| Hedge | $h$ | Annual volatility | Volatility in CHF |
|---|---|---|---|
| none | 0 | 18.0% | CHF 1,800,000 |
| 1:1 (83 contracts) | 1.00 | 6.0% | CHF 600,000 |
| minimum variance (95 contracts) | 1.14 | 5.6% | CHF 562,000 |

Hedge effectiveness $\rho^2 = 90.25\%$. The portfolio is more volatile than the index, so it
needs **more** than one unit of futures per unit of value.

### Example 3: a cross hedge

A CHF corporate bond portfolio hedged with Swiss government bond futures: $\sigma_S = 6\%$,
$\sigma_F = 5\%$, $\rho = 0.7$ (credit spreads move on their own).

$$
h^* = 0.7 \cdot \frac{6}{5} = 0.84,
\qquad
\text{effectiveness} = 0.49,
\qquad
\sigma_S\sqrt{1-0.49} = 4.3\%
$$

The best possible hedge removes only half the variance and leaves 4.3% of the original 6%
volatility. What remains is **basis risk**: the part of the spot that the hedge instrument
does not move with.

## 4. Finance Application

### Where $h^*$ is used

- **Equity beta hedging**: neutralize market exposure of a stock portfolio with index futures
  (Example 2), keeping only the stock-specific return.
- **Cross hedging**: when no future exists on the exact exposure (jet fuel, a corporate bond
  portfolio, a basket of mid caps), hedge with the closest liquid one (Example 3).
- **Currency overlays**: hedging a foreign equity position's FX risk, where the right ratio
  depends on how the asset and the currency co-move.

### What the numbers mean

- $h^*$ is a **regression slope**, estimated from historical returns. Practitioners compute
  it on daily or weekly returns over one to two years.
- $\rho^2$ is the **hedge effectiveness**. An effectiveness of 49% means half the variance
  survives the best possible hedge; that residual is basis risk, which no hedge ratio removes.
- $\sigma_S\sqrt{1-\rho^2}$ falls slowly as $\rho$ rises: $\rho = 0.9$ still leaves 44% of the
  volatility, $\rho = 0.99$ leaves 14%.

### The usual misreadings

- "A 1:1 hedge is the neutral, safe choice." It is optimal only when $\rho\,\sigma_S = \sigma_F$.
  It can even raise risk (01 showed this at low correlation).
- "Hedged means riskless." The minimum is $\sigma_S^2(1-\rho^2)$, zero only at $|\rho| = 1$.
- "The hedge ratio is a known constant." It is an estimate from noisy data and drifts over
  time; the hedge has to be re-estimated and rebalanced.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

rng = np.random.default_rng(42)
periods_per_year = 252

### Deriving $h^*$ with SymPy

In [ ]:
h_sym, rho_sym = sp.symbols("h rho", real=True)
s_sym, f_sym = sp.symbols("sigma_S sigma_F", positive=True)

V = s_sym**2 + h_sym**2 * f_sym**2 - 2 * h_sym * rho_sym * s_sym * f_sym
h_star = sp.solve(sp.diff(V, h_sym), h_sym)[0]

print("h*       =", h_star)
print("V''(h)   =", sp.diff(V, h_sym, 2))
print("V(h*)    =", sp.factor(sp.simplify(V.subs(h_sym, h_star))))   # = sigma_S^2 (1 - rho^2)

### Helpers

In [ ]:
def hedged_variance(h, sigma_s, sigma_f, rho):
    # Var(S - h F)
    return sigma_s**2 + h**2 * sigma_f**2 - 2 * h * rho * sigma_s * sigma_f


def min_variance_hedge_ratio(sigma_s, sigma_f, rho):
    return rho * sigma_s / sigma_f


def n_contracts(h, exposure_value, index_level, multiplier):
    # Futures contract value = index level * multiplier (CHF per index point)
    return h * exposure_value / (index_level * multiplier)

### Example 2 by formula

In [ ]:
portfolio_value = 10_000_000   # CHF
sigma_s, sigma_f, rho = 0.18, 0.15, 0.95
smi_level, multiplier = 12_000, 10

h_opt = min_variance_hedge_ratio(sigma_s, sigma_f, rho)
rows = []
for label, h_ratio in [("none", 0.0), ("1:1", 1.0), ("minimum variance", h_opt)]:
    vol = np.sqrt(hedged_variance(h_ratio, sigma_s, sigma_f, rho))
    rows.append({
        "hedge": label,
        "h": h_ratio,
        "contracts": n_contracts(h_ratio, portfolio_value, smi_level, multiplier),
        "annual vol": vol,
        "vol in CHF": vol * portfolio_value,
    })

pd.DataFrame(rows).set_index("hedge").round({"h": 3, "contracts": 1, "annual vol": 4, "vol in CHF": 0})

### Estimating $h^*$ from simulated daily returns

Simulate two years of daily returns with the parameters of Example 2 and estimate $h^*$ two
ways: as $\widehat{\operatorname{Cov}}/\widehat{\operatorname{Var}}$ and as a least-squares slope.

In [ ]:
def simulate_spot_futures(n, sigma_s, sigma_f, rho, rng):
    # Daily returns (decimals), zero mean, from annual volatilities
    daily_s = sigma_s / np.sqrt(periods_per_year)
    daily_f = sigma_f / np.sqrt(periods_per_year)
    cov = np.array([
        [daily_s**2, rho * daily_s * daily_f],
        [rho * daily_s * daily_f, daily_f**2],
    ])
    returns = rng.multivariate_normal(mean=[0.0, 0.0], cov=cov, size=n)
    return returns[:, 0], returns[:, 1]


def estimate_hedge_ratio(spot_r, fut_r):
    return np.cov(spot_r, fut_r)[0, 1] / np.var(fut_r, ddof=1)


spot_r, fut_r = simulate_spot_futures(2 * periods_per_year, sigma_s, sigma_f, rho, rng)

h_cov = estimate_hedge_ratio(spot_r, fut_r)
h_ols = np.polyfit(fut_r, spot_r, deg=1)[0]
residual = spot_r - h_cov * fut_r

print(f"h* (cov/var):        {h_cov:.4f}")
print(f"h* (OLS slope):      {h_ols:.4f}")
print(f"h* (true):           {h_opt:.4f}")
print(f"effectiveness (1 - Var(hedged)/Var(spot)): {1 - residual.var(ddof=1) / spot_r.var(ddof=1):.3f}   (rho^2 = {rho**2:.4f})")
print(f"corr(residual, futures): {np.corrcoef(residual, fut_r)[0, 1]:+.1e}")

The two estimates agree to machine precision: the minimum-variance hedge ratio **is** the OLS
slope. The residual is uncorrelated with the futures by construction.

Note the estimate itself: even with two years of daily data it can land visibly away from the
true 1.14 (this seed gives an unusually large miss). The next section measures how big the
typical estimation error is.

### How noisy is the estimated hedge ratio, and what does it cost?

Estimate $h^*$ on a window of past data, then apply it to the **next** year. Repeat many
times for a short window (3 months) and a long one (2 years).

In [ ]:
def out_of_sample_trial(n_estimation, rng, n_test=periods_per_year):
    s, f = simulate_spot_futures(n_estimation + n_test, sigma_s, sigma_f, rho, rng)
    h_est = estimate_hedge_ratio(s[:n_estimation], f[:n_estimation])
    s_test, f_test = s[n_estimation:], f[n_estimation:]
    effectiveness = 1 - np.var(s_test - h_est * f_test) / np.var(s_test)
    return h_est, effectiveness


n_trials = 2_000
results = {}
for label, n_est in [("63 days", 63), ("504 days", 504)]:
    trials = np.array([out_of_sample_trial(n_est, rng) for _ in range(n_trials)])
    results[label] = trials

summary = pd.DataFrame({
    label: {
        "mean h_est": t[:, 0].mean(),
        "std h_est": t[:, 0].std(),
        "mean out-of-sample effectiveness": t[:, 1].mean(),
        "5% worst effectiveness": np.quantile(t[:, 1], 0.05),
    }
    for label, t in results.items()
}).T
summary.round(4)

With 3 months of data the estimated ratio typically misses by a few percent. Because $V(h)$
is flat near its minimum, the cost in effectiveness is small. Estimation error matters more
when $\rho$ is low (flatter parabola, noisier slope) or when the true relationship drifts over
time, which this simulation does not include.

## 6. Visualizations

In [ ]:
h_grid = np.linspace(0, 1.6, 321)
vol_grid = np.sqrt(hedged_variance(h_grid, sigma_s, sigma_f, rho))

plt.figure(figsize=(8, 5))
plt.plot(h_grid, vol_grid, label="σ(S − hF)")
plt.axvline(h_opt, color="tab:green", linestyle="--", label=f"h* = {h_opt:.2f}")
plt.axvline(1.0, color="grey", linestyle=":", label="1:1 hedge")
plt.scatter([h_opt], [np.sqrt(hedged_variance(h_opt, sigma_s, sigma_f, rho))], color="tab:green", zorder=3)
plt.title("Example 2: hedged volatility as a function of the hedge ratio")
plt.xlabel("hedge ratio h (futures value per unit of portfolio value)")
plt.ylabel("annual volatility of hedged position")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
rho_grid = np.linspace(0, 1, 201)
sigma = 0.20   # equal volatilities

vol_optimal = np.sqrt(hedged_variance(min_variance_hedge_ratio(sigma, sigma, rho_grid), sigma, sigma, rho_grid))
vol_naive = np.sqrt(hedged_variance(1.0, sigma, sigma, rho_grid))

plt.figure(figsize=(8, 5))
plt.plot(rho_grid, vol_optimal / sigma, label="minimum-variance hedge  √(1 − ρ²)")
plt.plot(rho_grid, vol_naive / sigma, label="1:1 hedge  √(2(1 − ρ))")
plt.axhline(1.0, color="grey", linestyle="--", label="unhedged")
plt.title("Residual risk as a share of unhedged risk (equal volatilities)")
plt.xlabel("correlation ρ between spot and hedge")
plt.ylabel("hedged volatility / unhedged volatility")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for label, t in results.items():
    axes[0].hist(t[:, 0], bins=60, density=True, alpha=0.6, label=label)
    axes[1].hist(t[:, 1], bins=60, density=True, alpha=0.6, label=label)

axes[0].axvline(h_opt, color="black", linestyle="--", label="true h*")
axes[0].set_title("Estimated hedge ratio")
axes[0].set_xlabel("h estimate")
axes[1].axvline(rho**2, color="black", linestyle="--", label="ρ² (best possible)")
axes[1].set_title("Effectiveness in the following year")
axes[1].set_xlabel("1 − Var(hedged) / Var(spot)")

for ax in axes:
    ax.set_ylabel("density")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. A portfolio worth CHF 25,000,000 has 22% annual volatility and correlation 0.9 with SMI
   futures (15% volatility, SMI at 12,000, CHF 10 per point). Compute $h^*$, the number of
   contracts, and the residual volatility in CHF.
2. Spot and hedge have equal volatility and $\rho = 0.6$. Compare the hedged volatility of the
   1:1 hedge with the minimum-variance hedge.

### Intermediate

3. Show that the 1:1 hedge reduces variance (compared with no hedge) only if
   $\rho > \sigma_F / (2\sigma_S)$. Check the threshold numerically with `hedged_variance`.
4. Interpret: a treasurer reports "our hedge has an effectiveness of 64%". What is the
   correlation between the exposure and the hedge, and what share of the original
   **volatility** is left?

### Advanced

5. Instead of minimizing variance, a hedger minimizes $\operatorname{Var}(S - hF) - \lambda\,E[S - hF]$
   for some $\lambda > 0$, where the futures has expected return $\mu_F > 0$. Derive the optimal $h$.
   Why does it differ from $h^*$, and in which direction?

## 8. Solutions / Checks

In [ ]:
# 1.
value, s_vol, f_vol, corr = 25_000_000, 0.22, 0.15, 0.9
h1 = min_variance_hedge_ratio(s_vol, f_vol, corr)
print(f"1. h* = {h1:.3f}, contracts = {n_contracts(h1, value, 12_000, 10):.1f}, "
      f"residual vol = CHF {np.sqrt(hedged_variance(h1, s_vol, f_vol, corr)) * value:,.0f}")

# 2.
sigma, corr = 0.20, 0.6
print(f"2. 1:1 hedge vol = {np.sqrt(hedged_variance(1.0, sigma, sigma, corr)):.2%}, "
      f"min-variance vol = {np.sqrt(hedged_variance(corr, sigma, sigma, corr)):.2%}  (unhedged {sigma:.0%})")

# 3. Check the threshold: at rho = sigma_F / (2 sigma_S) the 1:1 hedge leaves variance unchanged
s_vol, f_vol = 0.10, 0.15
rho_threshold = f_vol / (2 * s_vol)
for corr in [rho_threshold - 0.05, rho_threshold, rho_threshold + 0.05]:
    print(f"3. rho = {corr:.3f}: Var(1:1) - Var(unhedged) = {hedged_variance(1.0, s_vol, f_vol, corr) - s_vol**2:+.5f}")

# 4.
effectiveness = 0.64
print(f"4. |rho| = {np.sqrt(effectiveness):.2f}, remaining volatility share = {np.sqrt(1 - effectiveness):.0%}")

**3.** $V(1) < \sigma_S^2 \iff \sigma_F^2 - 2\rho\,\sigma_S\sigma_F < 0 \iff \rho > \sigma_F/(2\sigma_S)$.
A hedge instrument twice as volatile as the spot needs $\rho > 1$, so a 1:1 hedge with it
**always** adds risk.

**4.** Effectiveness is $\rho^2 = 0.64$, so $|\rho| = 0.8$. The remaining volatility is
$\sqrt{1 - 0.64} = 60\%$ of the unhedged volatility: removing 64% of the **variance** removes
only 40% of the **volatility**. Effectiveness numbers sound better than the risk reduction
a reader feels.

**5.** With $E[S - hF] = \mu_S - h\mu_F$:

$$
\frac{d}{dh}\Big[\sigma_S^2 + h^2\sigma_F^2 - 2h\rho\sigma_S\sigma_F - \lambda(\mu_S - h\mu_F)\Big]
= 2h\sigma_F^2 - 2\rho\sigma_S\sigma_F + \lambda\mu_F = 0
$$

$$
h = h^* - \frac{\lambda\,\mu_F}{2\sigma_F^2}
$$

Shorting a future with positive expected return costs expected return, so a hedger who cares
about return hedges **less** than $h^*$. The minimum-variance ratio is the special case
$\lambda = 0$ (or $\mu_F = 0$).

## 9. Conclusion

- Hedged variance $V(h) = \sigma_S^2 + h^2\sigma_F^2 - 2h\rho\sigma_S\sigma_F$ is a parabola in $h$.
- Its minimum is at $h^* = \rho\,\sigma_S/\sigma_F = \operatorname{Cov}(S,F)/\operatorname{Var}(F)$: the regression beta.
- Best achievable variance $\sigma_S^2(1-\rho^2)$; hedge effectiveness $= \rho^2$. The rest is basis risk.
- Contracts: $N^* = h^*\,Q_S/Q_F$.
- The 1:1 hedge is optimal only by coincidence and can add risk; $h^*$ never does.
- $h^*$ is estimated from data. Near the optimum the variance curve is flat, so small estimation
  errors cost little; a drifting relationship costs more.

Next: [04 The two-asset frontier](./04_two_asset_frontier.ipynb).